# 03. Statistical Analysis

This notebook builds on the exploratory data analysis by applying statistical methods to quantify relationships and differences observed in the used-car dataset.

The analysis focuses on descriptive statistics, correlations, group comparisons, hypothesis testing, and simple linear regression.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from scipy import stats


def iter_start_paths():
    yield Path.cwd()

    session_name = os.environ.get("JPY_SESSION_NAME")
    if session_name:
        session_path = Path(session_name)
        if not session_path.is_absolute():
            session_path = (Path.cwd() / session_path).resolve()
        yield session_path.parent if session_path.suffix == ".ipynb" else session_path

    ipython_shell = globals().get("get_ipython", lambda: None)()
    if ipython_shell is not None:
        vscode_notebook = ipython_shell.user_ns.get("__vsc_ipynb_file__")
        if vscode_notebook:
            yield Path(vscode_notebook).resolve().parent


def find_project_root() -> Path:
    visited = set()
    for start_path in iter_start_paths():
        for candidate in [start_path, *start_path.parents]:
            if candidate in visited:
                continue
            visited.add(candidate)
            if (candidate / "data").exists() and (candidate / "notebooks").exists():
                return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and notebooks/")


PROJECT_ROOT = find_project_root()
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned_car_data.csv"

if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"{CLEAN_DATA_PATH} was not found. Run notebooks/01_data_inspection.ipynb first to generate it."
    )

df = pd.read_csv(CLEAN_DATA_PATH)

df.head()


## 01. Descriptive Statistics

These summaries establish the scale and spread of the numerical variables before formal tests are applied. The mean, median, standard deviation, and range also help identify skewed variables and possible outliers.

In [ ]:
numeric_columns = [
    "year",
    "selling_price",
    "km_driven",
    "car_age"
]

df[numeric_columns].describe().T

In [ ]:
summary = df[numeric_columns].agg(
    ["mean", "median", "std", "min", "max"]
).T

summary

## 02. Correlation with Selling Price

The correlations below identify which numerical variables move most closely with selling price. These are associations used to motivate the formal Pearson test and regression that follow.

In [ ]:
price_correlations = (
    df[numeric_columns]
    .corr()["selling_price"]
    .drop("selling_price")
    .sort_values(key=abs, ascending=False)
)

price_correlations

## 03. Transmission Price Difference

This section compares selling prices for manual and automatic vehicles. Welch's independent two-sample t-test is used because it does not assume equal variance between the groups, and Cohen's d measures the practical size of the difference.

In [ ]:
manual_prices = df.loc[
    df["transmission"] == "Manual",
    "selling_price"
]

automatic_prices = df.loc[
    df["transmission"] == "Automatic",
    "selling_price"
]

print(f"Manual vehicles: {len(manual_prices):,}")
print(f"Automatic vehicles: {len(automatic_prices):,}")
print(f"Manual mean price: {manual_prices.mean():,.0f}")
print(f"Automatic mean price: {automatic_prices.mean():,.0f}")

t_stat, p_value = stats.ttest_ind(
    manual_prices,
    automatic_prices,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("The difference is statistically significant at the 5% level.")
else:
    print("The difference is not statistically significant at the 5% level.")

In [ ]:
# Calculate Cohen's d effect size

manual_mean = manual_prices.mean()
automatic_mean = automatic_prices.mean()

manual_std = manual_prices.std()
automatic_std = automatic_prices.std()

n_manual = len(manual_prices)
n_automatic = len(automatic_prices)

pooled_std = np.sqrt(
    ((n_manual - 1) * manual_std**2 +
     (n_automatic - 1) * automatic_std**2)
    / (n_manual + n_automatic - 2)
)

cohens_d = (automatic_mean - manual_mean) / pooled_std

print("Cohen's d:", cohens_d)
print(
    f"Cohen's d = {cohens_d:.2f}, indicating a very large difference "
    "in selling prices between automatic and manual vehicles."
)

## 04. Hypothesis Testing: Car Age and Selling Price

This section tests whether there is a statistically significant linear relationship between car age and selling price.

- **Null hypothesis (H₀):** There is no linear correlation between car age and selling price in the population.
- **Alternative hypothesis (H₁):** There is a linear correlation between car age and selling price in the population.

A Pearson correlation test is used with a significance level of 5%.

In [ ]:
age_correlation, age_p_value = stats.pearsonr(
    df["car_age"],
    df["selling_price"]
)

print(f"Pearson correlation: {age_correlation:.3f}")
print(f"p-value: {age_p_value:.4g}")

### Hypothesis Testing Conclusion

The Pearson correlation test found a statistically significant negative linear relationship between car age and selling price (r = -0.424, p < 0.001).

This indicates that older vehicles tend to have lower selling prices in this dataset. The correlation is moderate rather than strong, meaning car age explains only part of the variation in selling prices.

The result shows an association and does not establish that increasing car age directly causes a decrease in selling price.

In [ ]:
regression = stats.linregress(
    df["car_age"],
    df["selling_price"]
)

print(f"Slope: {regression.slope:,.2f}")
print(f"Intercept: {regression.intercept:,.2f}")
print(f"R-squared: {regression.rvalue ** 2:.3f}")
print(f"p-value: {regression.pvalue:.4g}")

plt.figure(figsize=(10, 6))

sns.regplot(
    data=df,
    x="car_age",
    y="selling_price",
    scatter_kws={"alpha": 0.4},
    line_kws={"linewidth": 2}
)

plt.title("Car Age vs Selling Price - Linear Regression")
plt.xlabel("Car Age (years)")
plt.ylabel("Selling Price")
plt.show()

### Regression Conclusion

The simple linear regression model found a statistically significant negative relationship between car age and selling price.

The estimated regression slope indicates that each additional year of car age is associated with an estimated decrease of approximately ₹50,820 in selling price.

The model has an R² of approximately 0.18, meaning car age alone explains about 18% of the variation in selling prices. This indicates that other factors, such as vehicle model, mileage, fuel type, transmission, and seller characteristics, also contribute substantially to price variation.

The regression demonstrates an association rather than a causal relationship.

## 05. Statistical Findings

This section summarizes the main statistical results obtained throughout the analysis.

### Key Statistical Findings

- Selling prices are right-skewed, with a mean of approximately ₹473,913 and a median of ₹350,000.
- Car age has a moderate negative correlation with selling price (r = -0.424).
- The correlation between car age and selling price is statistically significant (p < 0.001).
- Automatic vehicles have a substantially higher mean selling price than manual vehicles in this dataset.
- The difference in mean selling prices between transmission groups is statistically significant (p < 0.001).
- The Cohen's d effect size for the transmission comparison is approximately 1.97, indicating a very large standardized difference.
- Simple linear regression estimates that each additional year of car age is associated with approximately ₹50,820 lower selling price.
- Car age alone explains approximately 18% of the variation in selling prices (R² = 0.18).
- These statistical results describe associations in the dataset and do not establish causal relationships.

## Conclusion

The statistical analysis provides evidence that vehicle characteristics are associated with differences in used-car selling prices.

Car age shows a moderate negative relationship with selling price, while transmission type is associated with a substantial difference in observed prices between automatic and manual vehicles. The regression analysis further shows that car age alone explains only a portion of the variation in selling prices, highlighting the influence of additional vehicle characteristics.

Overall, the analysis demonstrates how descriptive statistics, correlation analysis, hypothesis testing, effect size, and regression can be combined to extract meaningful insights from used-car data.